# Project: Precision Pharmacology Pipeline

## Phase 2: Relational Database Architecture

This file takes the separated CSVs generated in Phase 1 and ingests them into a fast, relational SQLite database using Primary and Foreign Keys.

### Step 1: Connect and Create the Database

In [1]:
# 1. Connect and Create the Database
import sqlite3
import pandas as pd

conn = sqlite3.connect('precision_pharmacology.db')
cursor = conn.cursor()

print("✅ Database created and connected.")

✅ Database created and connected.


### Step 2: Defining the Relational Schema & Indexes

We are building two tables:
1. `compound_metadata`: Holds the biological (MoA) and structural (SMILES) details of the drugs.
2. `genetic_signatures`: Holds the millions of experimental Z-scores.

In [2]:
# 1. Drop the old tables so we can start fresh
cursor.execute('DROP TABLE IF EXISTS compound_metadata')
cursor.execute('DROP TABLE IF EXISTS genetic_signatures')
cursor.execute('DROP TABLE IF EXISTS drug_signatures') # Drop the old flat one just in case

# 2. Create Table 1: Compound Metadata (The "Left" Table)
cursor.execute('''
    CREATE TABLE compound_metadata (
        pert_iname TEXT PRIMARY KEY,
        moa TEXT,
        target TEXT,
        canonical_smiles TEXT,
        pubchem_cid TEXT
    )
''')

# 3. Create Table 2: Genetic Signatures (The "Right" Table)
cursor.execute('''
    CREATE TABLE genetic_signatures (
        pert_iname TEXT,
        sig_id TEXT,
        pr_gene_symbol TEXT,
        z_score FLOAT,
        is_significant INTEGER,
        FOREIGN KEY (pert_iname) REFERENCES compound_metadata(pert_iname)
    )
''')

# 4. Create Indexes (Crucial for fast JOINs and searching)
cursor.execute('CREATE INDEX idx_meta_pert ON compound_metadata(pert_iname)')
cursor.execute('CREATE INDEX idx_sig_pert ON genetic_signatures(pert_iname)')
cursor.execute('CREATE INDEX idx_gene_symbol ON genetic_signatures(pr_gene_symbol)')
cursor.execute('CREATE INDEX idx_zscore ON genetic_signatures(z_score)')

conn.commit()
print("✅ Relational Schema and Indexes successfully created.")

✅ Relational Schema and Indexes successfully created.


### Step 3: The Multi-File "Bulk Load"
We load the lightweight metadata first, then stream the massive signatures file in chunks to prevent RAM crashes.

In [3]:
# Step 3: The "Bulk Load"

# Upload Metadata
print("Uploading Compound Metadata...")
df_meta = pd.read_csv('lincs_compound_metadata.csv')
df_meta.to_sql('compound_metadata', conn, if_exists='append', index=False)
print(f"✅ Uploaded {len(df_meta)} unique drugs to compound_metadata.")

# Upload Signatures (Streaming in chunks of 100,000)
print("\nUploading Genetic Signatures...")
df_sigs = pd.read_csv('lincs_genetic_signatures.csv')
df_sigs.to_sql('genetic_signatures', conn, if_exists='append', index=False, chunksize=100000)
print(f"✅ Successfully uploaded {len(df_sigs)} expression rows to genetic_signatures.")

Uploading Compound Metadata...
✅ Uploaded 5321 unique drugs to compound_metadata.

Uploading Genetic Signatures...
✅ Successfully uploaded 1049972 expression rows to genetic_signatures.


### Step 4: The Final Verification (Testing the JOIN)

In [4]:
# The Final Verification
# Run this to make sure the database is alive and the relational JOIN works.

# Relational SQL Test: Count top 5 strongest 'up-regulated' hits AND their MoA
test_query = """
SELECT 
    m.pert_iname, 
    m.moa, 
    s.pr_gene_symbol, 
    s.z_score 
FROM compound_metadata m
JOIN genetic_signatures s 
    ON m.pert_iname = s.pert_iname
ORDER BY s.z_score DESC 
LIMIT 5
"""

print("Testing Relational JOIN Query:")
print(pd.read_sql(test_query, conn))

conn.close()
print("\n✅ Database connection closed safely.")

Testing Relational JOIN Query:
        pert_iname             moa pr_gene_symbol    z_score
0  rhodomyrtoxin-b            None          DDIT4  10.000002
1     geldanamycin   HSP inhibitor         FKBP14  10.000001
2         NSC-3852  HDAC inhibitor          ABHD4  10.000001
3   trichostatin-a  HDAC inhibitor          KIF5C  10.000001
4    BRD-K00816165            None         KLHL21  10.000001

✅ Database connection closed safely.
